In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import pandas as pd
matplotlib.rcParams['font.family'] = 'Times New Roman'  # 修改全局字体为Times New Roman
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
# matplotlib.rcParams['savefig.dpi'] = 300  # 设置保存图片的dpi为300

In [ ]:
# -------------------------------
# 1. 加载功能连接矩阵 & 向量化
# -------------------------------
def load_fc_matrices_with_filenames(folder):
    files = sorted([f for f in os.listdir(folder) if f.endswith('.csv')])
    data = []
    filenames = []
    for f in files:
        mat = np.loadtxt(os.path.join(folder, f), delimiter=',')
        print(f"Processing file: {f}, shape: {mat.shape}")
        vec = mat[np.triu_indices_from(mat, k=1)]
        data.append(vec)
        filenames.append(f)  # 记录文件名
    return np.array(data), filenames

# 加载数据并记录文件名
X_ADHD, filenames_ADHD = load_fc_matrices_with_filenames(r'E:\aliyun_backup\muilt_disorders\10_combat\ADHD')
X_ASD, filenames_ASD = load_fc_matrices_with_filenames(r'E:\aliyun_backup\muilt_disorders\10_combat\ASD')
X_SCZ, filenames_SCZ = load_fc_matrices_with_filenames(r'E:\aliyun_backup\muilt_disorders\10_combat\SCZ')
X_HC, filenames_HC = load_fc_matrices_with_filenames(r'E:\aliyun_backup\muilt_disorders\10_combat\HC')

# 合并数据和文件名
original_filenames = filenames_HC + filenames_ADHD + filenames_ASD + filenames_SCZ

In [ ]:
# -------------------------------
# 2. 构建标签和数据集合
# -------------------------------
X = np.concatenate([X_HC, X_ADHD, X_ASD, X_SCZ], axis=0)
y = np.array([0]*len(X_HC) + [1]*len(X_ADHD) + [2]*len(X_ASD) + [3]*len(X_SCZ))  # 0=HC, 1=A, 2=B, 3=C

In [ ]:
X.shape, y.shape

In [ ]:
# 构建 ND (A+B+C) vs HC 二分类标签用于共性模式提取
nd_idx = np.where((y == 1) | (y == 2) | (y == 3))[0]
hc_idx = np.where(y == 0)[0]
X_ndhc = np.concatenate([X[hc_idx], X[nd_idx]], axis=0)
y_ndhc = np.array([0]*len(hc_idx) + [1]*len(nd_idx))

In [ ]:
# -------------------------------
# 3. 标准化并提取共性异常方向 (PLS)
# -------------------------------
scaler = StandardScaler()
X_ndhc_std = scaler.fit_transform(X_ndhc)

pls = PLSRegression(n_components=1)
pls.fit(X_ndhc_std, y_ndhc)
shared_direction = pls.x_weights_[:, 0]  # 共性异常向量

In [ ]:
# -------------------------------
# 7. 连接模式反投影（Top N贡献连接）
# -------------------------------
N = int((1 + np.sqrt(1 + 8 * X.shape[1])) // 2)  # 推导原始矩阵大小
print(f"Original matrix size: {N}x{N}")
fc_map = np.zeros((N, N))
fc_map[np.triu_indices(N, k=1)] = shared_direction
fc_map = fc_map + fc_map.T  # 对称化

K = int(0.01 * len(shared_direction))  # 取前1%的连接
abs_weights = np.abs(shared_direction)
top_idx = np.argsort(abs_weights)[-K:]
top_conn = np.zeros_like(shared_direction)
top_conn[top_idx] = shared_direction[top_idx]

top_map = np.zeros((N, N))
top_map[np.triu_indices(N, k=1)] = top_conn
top_map = top_map + top_map.T

# 保存连接模式反投影数据为CSV文件
# pd.DataFrame(top_map).to_csv(r"E:\aliyun_backup\muilt_disorders\11_pls\result\pls_weight\偏离连接_ALL_1.csv", index=False, header=False)

# 可视化 TopK 异常连接
plt.figure(figsize=(6, 5))
sns.heatmap(top_map, cmap='coolwarm', center=0, square=True, vmin=-0.01, vmax=0.01)

plt.xlabel('Brain Region', fontsize=16)
plt.ylabel('Brain Region', fontsize=16)
plt.tight_layout()
# plt.savefig(r'E:\aliyun_backup\muilt_disorders\11_pls\result\residual_100_plot_contribute_all.tif', dpi=300)
plt.show()

In [ ]:
# -------------------------------
# 4. 排除HC组并进行投影 & 残差计算
# -------------------------------
N = int((1 + np.sqrt(1 + 8 * X.shape[1])) // 2)  # 推导原始矩阵大小
print(f"Original matrix size: {N}x{N}")

# 提前排除HC组
non_hc_idx = np.where(y != 0)[0]  # 非HC组索引
X_non_hc = X[non_hc_idx]  # 只保留非HC组数据

# 对非HC组数据进行标准化
X_non_hc_std = scaler.transform(X_non_hc)
projections = X_non_hc_std @ shared_direction
reconstructed = np.outer(projections, shared_direction)

# 计算残差
residuals = X_non_hc_std - reconstructed

# 将残差还原为功能连接矩阵并保存
output_base_path = r'E:\aliyun_backup\muilt_disorders\11_residual'
labels = ['ADHD', 'ASD', 'SCZ']

In [ ]:
residuals.shape

In [ ]:
# -------------------------------
# 5. PCA 降维残差空间
# -------------------------------
pca = PCA(n_components=2)
residual_pca = pca.fit_transform(residuals)

# 将共有异常向量投影到PCA空间
shared_direction_pca = pca.transform([shared_direction])[0]  # 投影到PCA空间

In [ ]:
# -------------------------------
# 8. 各病种残差方向的余弦相似度
# -------------------------------
def compute_mean_direction(idx):
    vec = residuals[idx]
    mean = vec.mean(axis=0)
    return mean / np.linalg.norm(mean)

In [ ]:
dir_ADHD = compute_mean_direction(np.where(y[non_hc_idx] == 1)[0])
dir_ASD = compute_mean_direction(np.where(y[non_hc_idx] == 2)[0])
dir_SCZ = compute_mean_direction(np.where(y[non_hc_idx] == 3)[0])

cos_ADHD_ASD = cosine_similarity([dir_ADHD], [dir_ASD])[0,0]
cos_ADHD_SCZ = cosine_similarity([dir_ADHD], [dir_SCZ])[0,0]
cos_ASD_SCZ = cosine_similarity([dir_ASD], [dir_SCZ])[0,0]

print("病种偏离方向的余弦相似度：")
print(f"ADHD vs ASD: {cos_ADHD_ASD:.3f}")
print(f"ADHD vs SCZ: {cos_ADHD_SCZ:.3f}")
print(f"ASD vs SCZ: {cos_ASD_SCZ:.3f}")

In [ ]:
# -------------------------------
# 9. 每个病种的偏离方向贡献连接（Top K）
# -------------------------------
def extract_top_connection_map(residual_vec, top_k):
    abs_weights = np.abs(residual_vec)
    top_idx = np.argsort(abs_weights)[-top_k:]
    top_conn = np.zeros_like(residual_vec)
    top_conn[top_idx] = residual_vec[top_idx]

    # 转换为对称矩阵形式
    fc_top_map = np.zeros((N, N))
    fc_top_map[np.triu_indices(N, k=1)] = top_conn
    fc_top_map = fc_top_map + fc_top_map.T
    return fc_top_map

# 计算每个病种的平均残差方向
mean_resid_ADHD = residuals[np.where(y[non_hc_idx] == 1)[0]].mean(axis=0)
mean_resid_ASD = residuals[np.where(y[non_hc_idx] == 2)[0]].mean(axis=0)
mean_resid_SCZ = residuals[np.where(y[non_hc_idx] == 3)[0]].mean(axis=0)

# 提取前1%偏离连接
top_k = int(0.01 * len(mean_resid_ADHD))
print(f"Top K connections: {top_k}")

top_map_ADHD = extract_top_connection_map(mean_resid_ADHD, top_k)
top_map_ASD = extract_top_connection_map(mean_resid_ASD, top_k)
top_map_SCZ = extract_top_connection_map(mean_resid_SCZ, top_k)


In [ ]:
pd.DataFrame(top_map_ASD)

In [ ]:
pd.DataFrame(top_map_ADHD).to_csv(r"E:\aliyun_backup\muilt_disorders\11_pls\result\pls_weight\偏离连接_ADHD_1.csv", index=False, header=False)
pd.DataFrame(top_map_ASD).to_csv(r"E:\aliyun_backup\muilt_disorders\11_pls\result\pls_weight\偏离连接_ASD_1.csv", index=False, header=False)
pd.DataFrame(top_map_SCZ).to_csv(r"E:\aliyun_backup\muilt_disorders\11_pls\result\pls_weight\偏离连接_SCZ_1.csv", index=False, header=False)